# Datasets & Vocabulary

## Install required extensions

- `nltk` - for downloading the brown dataset
- `datasets` - hugging face tool for easily handling datasets

In [40]:
%pip install nltk conllu "datasets<4.0.0"

Note: you may need to restart the kernel to use updated packages.


## Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import nltk
from nltk.corpus import brown
from datasets import load_dataset

from src.data.vocabulary import (
    preprocess_tokens,
    preprocess_tags,
    build_vocab,
    Encoding,
    Vocabulary,
)


## Prepare datasets

In this section, we load the Brown Corpus and Universal Dependencies datasets. 
- The Brown Corpus provides POS-tagged sentences using the Brown tagset
- Universal Dependencies offers POS-tagged sentences using the universal POS tags. 

Both datasets will be used to build separate vocabularies for training sequence labeling models.

In [42]:
brown_sentences = brown.tagged_sents()
ud = load_dataset("universal_dependencies", "en_ewt")
nltk.download("brown")

[nltk_data] Downloading package brown to
[nltk_data]     /home/dangavriluta/nltk_data...
[nltk_data]   Package brown is already up-to-date!


True

In [43]:
# Test if working
print("Brown Corpus Sample:\n")
for w, t in brown_sentences[0]:
    print(f"{w:15} {t}")

print("\nUniversal Dependencies Sample:\n")
feature = ud["train"].features["upos"].feature
label_names = feature.names

tokens = ud["train"][0]["tokens"]
upos_ids = ud["train"][0]["upos"]

for w, t in zip(tokens, upos_ids):
    print(f"{w:15} {label_names[t]}")

Brown Corpus Sample:

The             AT
Fulton          NP-TL
County          NN-TL
Grand           JJ-TL
Jury            NN-TL
said            VBD
Friday          NR
an              AT
investigation   NN
of              IN
Atlanta's       NP$
recent          JJ
primary         NN
election        NN
produced        VBD
``              ``
no              AT
evidence        NN
''              ''
that            CS
any             DTI
irregularities  NNS
took            VBD
place           NN
.               .

Universal Dependencies Sample:

Al              PROPN
-               PUNCT
Zaman           PROPN
:               PUNCT
American        ADJ
forces          NOUN
killed          VERB
Shaikh          PROPN
Abdullah        PROPN
al              PROPN
-               PUNCT
Ani             PROPN
,               PUNCT
the             DET
preacher        NOUN
at              ADP
the             DET
mosque          NOUN
in              ADP
the             DET
town            NOUN
of      

## Build vocabularies

In this section, we build vocabularies from both the Brown Corpus and Universal Dependencies datasets. 

We create mappings between words/tags and their integer IDs, which will be used for encoding sentences into a format suitable for machine learning models.

## Datasets -> Vocabularies

### Brown dataset

In [55]:
# Preprocess the Brown dataset
preprocessed_brown = []

for sent in brown_sentences:
    tokens, tags = zip(*sent)          # unzip into two parallel tuples
    tokens = preprocess_tokens(tokens)
    tags   = preprocess_tags(tags)
    preprocessed_brown.append((tokens, tags))

# Build the vocabulary
vocab_brown = build_vocab(preprocessed_brown)

# Encode the dataset
encoded_brown = [vocab_brown.encode(tokens, tags) for tokens, tags in preprocessed_brown]

### Universal dependencies dataset

In [48]:
# Preprocess the Universal Dependencies dataset
preprocessed_ud = []

for item in ud["train"].select(range(1000)):
    tokens = preprocess_tokens(item["tokens"])
    tags   = [label_names[i] for i in item["upos"]]
    preprocessed_ud.append((tokens, tags))

# Build the vocabulary for UD
vocab_ud = build_vocab(preprocessed_ud)

# Encode the UD dataset
encoded_ud = [vocab_ud.encode(tokens, tags) for tokens, tags in preprocessed_ud]

In [51]:
def show_sample(name: str, encoding: Encoding, vocab: Vocabulary) -> None:
    tokens, tags = vocab.decode(encoding)
    print(f"\n{name} sample")
    print("-" * (len(name) + 7))
    print(f"encoded word ids : {encoding.word_ids}")
    print(f"encoded tag ids  : {encoding.tag_ids}")
    print("decoded tokens   :", " ".join(tokens))
    print("decoded tags     :", " ".join(tags))

show_sample("UD", encoded_ud[2], vocab_ud)
show_sample("Brown", encoded_brown[2], vocab_brown)



UD sample
---------
encoded word ids : [1270, 144, 2144, 471, 347, 4002, 4019, 1783, 683, 4214, 103, 3986, 746, 2845, 2024, 502, 24]
encoded tag ids  : [12, 13, 1, 8, 16, 14, 11, 4, 16, 2, 9, 1, 8, 16, 2, 12, 13]
decoded tokens   : dpa : iraqi authorities announced that they had busted up 3 terrorist cells operating in baghdad .
decoded tags     : PROPN PUNCT ADJ NOUN VERB SCONJ PRON AUX VERB ADP NUM ADJ NOUN VERB ADP PROPN PUNCT

Brown sample
------------
encoded word ids : [3857, 3477, 3835, 2184, 1856, 585, 802, 712, 1731, 3759, 1071, 2173, 1347, 3069, 3905, 2119, 3238, 2666, 2925, 217, 2128, 60, 2024, 3857, 1874, 2973, 4191, 4145, 4228, 712, 2419, 2142, 348, 2172, 65]
encoded tag ids  : [14, 56, 52, 52, 41, 21, 88, 45, 56, 46, 52, 52, 56, 56, 83, 85, 54, 45, 46, 95, 54, 3, 45, 14, 46, 52, 90, 17, 88, 45, 52, 56, 56, 56, 8]
decoded tokens   : the september-october term jury had been charged by fulton superior court judge durwood pye to investigate reports of possible `` irregularit